# Create a full pytorch dataset

In [1]:
import pandas as pd

df = pd.read_csv('raw.csv', delimiter=';')

/tmp/ipykernel_35427/386006593.py:3: DtypeWarning: Columns (163,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('raw.csv', delimiter=';')


In [2]:
df.columns
twos = 0
qcols = 0
for col in df.columns:
    if col.startswith('Q'):
        for val in df[col]:
            if val == 2:
                twos += 1

        qcols += 1
        
print(qcols)
print(twos)


36
1800


In [3]:
for col in df.columns:
    print(col)

NUM_POSTE
NOM_USUEL
LAT
LON
ALTI
AAAAMM
RR
QRR
NBRR
RR_ME
RRAB
QRRAB
RRABDAT
NBJRR1
NBJRR5
NBJRR10
NBJRR30
NBJRR50
NBJRR100
PMERM
QPMERM
NBPMERM
PMERMINAB
QPMERMINAB
PMERMINABDAT
TX
QTX
NBTX
TX_ME
TXAB
QTXAB
TXDAT
TXMIN
QTXMIN
TXMINDAT
NBJTX0
NBJTX25
NBJTX30
NBJTX35
NBJTXI20
NBJTXI27
NBJTXS32
TN
QTN
NBTN
TN_ME
TNAB
QTNAB
TNDAT
TNMAX
QTNMAX
TNMAXDAT
NBJTN5
NBJTN10
NBJTNI10
NBJTNI15
NBJTNI20
NBJTNS20
NBJTNS25
NBJGELEE
TAMPLIM
QTAMPLIM
TAMPLIAB
QTAMPLIAB
TAMPLIABDAT
NBTAMPLI
TM
QTM
NBTM
TMM
QTMM
NBTMM
NBJTMS24
TMMIN
QTMMIN
TMMINDAT
TMMAX
QTMMAX
TMMAXDAT
UNAB
QUNAB
UNABDAT
NBUN
UXAB
QUXAB
UXABDAT
NBUX
UMM
QUMM
NBUM
TSVM
QTSVM
NBTSVM
ETP
QETP
FXIAB
QFXIAB
DXIAB
QDXIAB
FXIDAT
NBJFF10
NBJFF16
NBJFF28
NBFXI
FXI3SAB
QFXI3SAB
DXI3SAB
QDXI3SAB
FXI3SDAT
NBJFXI3S10
NBJFXI3S16
NBJFXI3S28
NBFXI3S
FXYAB
QFXYAB
DXYAB
QDXYAB
FXYABDAT
NBJFXY8
NBJFXY10
NBJFXY15
NBFXY
FFM
QFFM
NBFFM
INST
QINST
NBINST
NBSIGMA0
NBSIGMA20
NBSIGMA80
GLOT
QGLOT
NBGLOT
DIFT
QDIFT
NBDIFT
DIRT
QDIRT
NBDIRT
HNEIGEFTOT
QHNEIGEFTOT
H

In [4]:
# Cell 1 — Build monthly date and filter to 2001-01 through 2025-12

import pandas as pd
import numpy as np

df_proc = df.copy()

# Robust monthly date from AAAAMM (expected format yyyymm, sometimes numeric)
df_proc["AAAAMM"] = df_proc["AAAAMM"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)

df_proc["month"] = pd.to_datetime(
    df_proc["AAAAMM"],
    format="%Y%m",
    errors="coerce"
).dt.to_period("M")

# Keep only the modelling period: 2001-01 to 2025-12 inclusive
start_month = pd.Period("2001-01", freq="M")
end_month = pd.Period("2025-12", freq="M")

rows_before = len(df_proc)
df_proc = df_proc[
    (df_proc["month"] >= start_month) &
    (df_proc["month"] <= end_month)
].copy()

# Basic sorting for later station-wise processing
df_proc = df_proc.sort_values(["NUM_POSTE", "month"]).reset_index(drop=True)

print("Rows before filter:", rows_before)
print("Rows after filter:", len(df_proc))
print("Unique NUM_POSTE:", df_proc["NUM_POSTE"].nunique())
print("Min month:", df_proc["month"].min())
print("Max month:", df_proc["month"].max())

df_proc[["NUM_POSTE", "AAAAMM", "month"]].head()

Rows before filter: 3262791
Rows after filter: 1044600
Unique NUM_POSTE: 5329
Min month: 2001-01
Max month: 2025-12


,NUM_POSTE,AAAAMM,month
0,1014002.0,200403,2004-03
1,1014002.0,200404,2004-04
2,1014002.0,200405,2004-05
3,1014002.0,200406,2004-06
4,1014002.0,200407,2004-07


In [5]:
1+1

2

In [6]:
# Cell 1.1 — Missingness report by column

import pandas as pd
import numpy as np

df_check = df_proc.copy()

missing_report = (
    df_check.isna()
    .sum()
    .to_frame("n_missing")
)

missing_report["pct_missing"] = missing_report["n_missing"] / len(df_check) * 100
missing_report["dtype"] = df_check.dtypes.astype(str)

missing_report = missing_report.sort_values(
    by=["pct_missing", "n_missing"],
    ascending=False
)

print("Total rows:", len(df_check))
print("Total columns:", df_check.shape[1])

# Show the worst offenders
display(missing_report.head(50))

# Optional: only columns with any missingness
cols_with_missing = missing_report[missing_report["n_missing"] > 0]
print("\nColumns with missing values:", len(cols_with_missing))

# Optional: quick view of the main model columns only
model_cols = [c for c in df_check.columns if c not in ["month"]]
model_missing = (
    df_check[model_cols]
    .isna()
    .sum()
    .to_frame("n_missing")
)
model_missing["pct_missing"] = model_missing["n_missing"] / len(df_check) * 100
model_missing = model_missing.sort_values("pct_missing", ascending=False)

print("\nTop missing model columns:")
display(model_missing.head(50))

Total rows: 1044600
Total columns: 167


,n_missing,pct_missing,dtype
code_departement,1044600,100.000000,float64
nom_departement,1044600,100.000000,object
code_region,1044600,100.000000,float64
nom_region,1044600,100.000000,object
DIFT,1043756,99.919204,float64
QDIFT,1043756,99.919204,float64
TX_ME,1043316,99.877082,float64
TN_ME,1043313,99.876795,float64
DIRT,1042838,99.831323,float64
QDIRT,1042838,99.831323,float64



Columns with missing values: 160

Top missing model columns:


,n_missing,pct_missing
code_region,1044600,100.000000
nom_region,1044600,100.000000
code_departement,1044600,100.000000
nom_departement,1044600,100.000000
QDIFT,1043756,99.919204
DIFT,1043756,99.919204
TX_ME,1043316,99.877082
TN_ME,1043313,99.876795
QDIRT,1042838,99.831323
DIRT,1042838,99.831323


In [7]:
# Cell 2 — Clean columns, process Q flags, keep >=97% complete features

import pandas as pd
import numpy as np

df_proc = df_proc.copy()

# -----------------------------
# 0) Drop textual / metadata columns we do not want
# -----------------------------
drop_text_cols = [
    "NOM_USUEL", "nom_usuel",
    "nom_region", "NOM_REGION",
    "nom_departement", "nom_department", "NOM_DEPARTEMENT",
    "code_region", "CODE_REGION",
]
df_proc = df_proc.drop(columns=[c for c in drop_text_cols if c in df_proc.columns], errors="ignore")

# -----------------------------
# 1) Clean NUM_POSTE and derive code_departement
# -----------------------------
df_proc["NUM_POSTE"] = (
    df_proc["NUM_POSTE"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\s+", "", regex=True)
)

def derive_code_departement(num_poste):
    s = "".join(ch for ch in str(num_poste) if ch.isdigit())
    if len(s) == 0:
        return pd.NA
    s = s.zfill(8)
    # Overseas / special codes
    if s.startswith(("97", "98", "99")):
        return s[:3]
    return s[:2]

df_proc["code_departement"] = df_proc["NUM_POSTE"].apply(derive_code_departement).astype("string")

# -----------------------------
# 2) Target columns
# -----------------------------
target_cols = ["TN", "TX"]

for tc in target_cols:
    if tc not in df_proc.columns:
        raise ValueError(f"Target column {tc} not found in dataframe.")

# Ensure month exists from Cell 1
if "month" not in df_proc.columns:
    raise ValueError("month column missing. Run Cell 1 first.")

# Sort and remove duplicate station-month rows if any
df_proc = df_proc.sort_values(["NUM_POSTE", "month"]).reset_index(drop=True)

n_dupes = df_proc.duplicated(subset=["NUM_POSTE", "month"]).sum()
print("Duplicate NUM_POSTE-month rows before:", int(n_dupes))

if n_dupes > 0:
    df_proc = (
        df_proc
        .sort_values(["NUM_POSTE", "month"])
        .drop_duplicates(subset=["NUM_POSTE", "month"], keep="first")
        .copy()
    )

df_proc = df_proc.sort_values(["NUM_POSTE", "month"]).reset_index(drop=True)

# -----------------------------
# 3) Process all Q columns:
#    if Q == 2 -> impute paired value from same month last/next year
#    then Q becomes binary: 1 if flagged, else 0
# -----------------------------
q_cols = [c for c in df_proc.columns if c.startswith("Q") and len(c) > 1]

def seasonal_impute_for_station_group(g, value_col, q_col):
    g = g.sort_values("month").copy()

    # numeric conversion for the value column
    g[value_col] = pd.to_numeric(g[value_col], errors="coerce")

    months = pd.PeriodIndex(g["month"], freq="M")
    values = g[value_col].to_numpy(dtype="float64")

    lookup = pd.Series(values, index=months)

    prev_vals = lookup.reindex(months - 12).to_numpy()
    next_vals = lookup.reindex(months + 12).to_numpy()

    both_available = ~pd.isna(prev_vals) & ~pd.isna(next_vals)
    only_prev = ~pd.isna(prev_vals) & pd.isna(next_vals)
    only_next = pd.isna(prev_vals) & ~pd.isna(next_vals)

    fill_values = np.full(len(g), np.nan, dtype="float64")
    fill_values[both_available] = (prev_vals[both_available] + next_vals[both_available]) / 2.0
    fill_values[only_prev] = prev_vals[only_prev]
    fill_values[only_next] = next_vals[only_next]

    q_mask = pd.to_numeric(g[q_col], errors="coerce").eq(2).fillna(False).to_numpy()

    # Fill only where the flag says Q == 2 and a fill value exists
    can_fill = q_mask & ~pd.isna(fill_values)
    g.loc[can_fill, value_col] = fill_values[can_fill]

    # Convert Q column to binary:
    # 1 = was flagged (Q == 2), 0 = everything else
    g[q_col] = 0
    g.loc[q_mask, q_col] = 1

    return g

processed = 0
skipped = []

for q_col in q_cols:
    value_col = q_col[1:]  # QRR -> RR, QTMM -> TMM, etc.

    if value_col not in df_proc.columns:
        skipped.append((q_col, value_col))
        continue

    # Apply station-wise to preserve seasonality
    df_proc = (
        df_proc
        .groupby("NUM_POSTE", group_keys=False)
        .apply(lambda g: seasonal_impute_for_station_group(g, value_col, q_col))
        .reset_index(drop=True)
    )

    processed += 1

print("\nQ columns found:", len(q_cols))
print("Q/value pairs processed:", processed)
print("Q columns skipped because base column missing:", len(skipped))
if skipped:
    print("Examples skipped:", skipped[:10])

# -----------------------------
# 4) Keep only columns that are >=97% complete
#    - always keep identifiers / targets / month
#    - keep Q columns only if their base variable is kept
# -----------------------------
missing_pct = df_proc.isna().mean() * 100

mandatory_cols = ["NUM_POSTE", "code_departement", "month"] + target_cols
exclude_from_threshold = set(mandatory_cols + ["AAAAMM"])

# Base columns = non-Q columns eligible by completeness
base_candidate_cols = [
    c for c in df_proc.columns
    if c not in exclude_from_threshold
    and not c.startswith("Q")
]

keep_base_cols = []
for col in base_candidate_cols:
    if pd.api.types.is_numeric_dtype(df_proc[col]) or col in ["NUM_POSTE", "code_departement"]:
        if missing_pct.get(col, 100.0) <= 3.0:
            keep_base_cols.append(col)

# Force targets to be kept
for tc in target_cols:
    if tc in df_proc.columns and tc not in keep_base_cols:
        keep_base_cols.append(tc)

# Keep Q columns only if their base variable is kept
keep_q_cols = []
for q_col in q_cols:
    base_col = q_col[1:]
    if base_col in keep_base_cols and q_col in df_proc.columns:
        keep_q_cols.append(q_col)

print("\nBase columns kept:", len(keep_base_cols))
print("Q columns kept:", len(keep_q_cols))

# -----------------------------
# 5) Impute remaining missing values for kept base numeric columns
#    using same month last/next year.
#    No extra imputation flags are created.
# -----------------------------
def seasonal_impute_missing(df_in, col):
    df_out = df_in.copy()
    df_out[col] = pd.to_numeric(df_out[col], errors="coerce")

    for station, idx in df_out.groupby("NUM_POSTE").groups.items():
        idx = list(idx)
        g = df_out.loc[idx].sort_values("month").copy()

        months = pd.PeriodIndex(g["month"], freq="M")
        values = g[col].to_numpy(dtype="float64")
        lookup = pd.Series(values, index=months)

        prev_vals = lookup.reindex(months - 12).to_numpy()
        next_vals = lookup.reindex(months + 12).to_numpy()

        both_available = ~pd.isna(prev_vals) & ~pd.isna(next_vals)
        only_prev = ~pd.isna(prev_vals) & pd.isna(next_vals)
        only_next = pd.isna(prev_vals) & ~pd.isna(next_vals)

        fill_values = np.full(len(g), np.nan, dtype="float64")
        fill_values[both_available] = (prev_vals[both_available] + next_vals[both_available]) / 2.0
        fill_values[only_prev] = prev_vals[only_prev]
        fill_values[only_next] = next_vals[only_next]

        missing_mask = g[col].isna().to_numpy()
        can_fill = missing_mask & ~pd.isna(fill_values)

        df_out.loc[g.index[can_fill], col] = fill_values[can_fill]

    return df_out

# Only impute retained base numeric columns, not identifiers or Q flags
impute_cols = [
    c for c in keep_base_cols
    if c not in ["NUM_POSTE", "code_departement", "month"] + target_cols
    and c in df_proc.columns
    and pd.api.types.is_numeric_dtype(df_proc[c])
]

# IMPORTANT: do NOT impute the targets here
# We'll keep TN and TX as targets and drop rows where they are missing.

print("\nImputing retained base numeric columns:")
for col in impute_cols:
    before = int(df_proc[col].isna().sum())
    if before > 0:
        df_proc = seasonal_impute_missing(df_proc, col)
    after = int(df_proc[col].isna().sum())
    print(f"{col}: missing before={before}, after={after}")

# -----------------------------
# 6) Final retained columns
# -----------------------------
imputed_cols = [c for c in df_proc.columns if c.endswith("_imputed")]  # should be empty here
final_cols = list(dict.fromkeys(
    mandatory_cols
    + keep_base_cols
    + keep_q_cols
    + imputed_cols
))

# Remove AAAAMM if present
final_cols = [c for c in final_cols if c in df_proc.columns and c != "AAAAMM"]

df_proc = df_proc[final_cols].copy()

# -----------------------------
# 7) Drop any rows with remaining missing in kept columns
#    This includes dropping rows where TN/TX are missing.
# -----------------------------
remaining_missing_rows = df_proc.isna().any(axis=1).sum()
print("\nRows with remaining missing in final kept columns:", int(remaining_missing_rows))

if remaining_missing_rows > 0:
    df_proc = df_proc.loc[~df_proc.isna().any(axis=1)].copy()

df_proc = df_proc.sort_values(["NUM_POSTE", "month"]).reset_index(drop=True)

print("\nFinal shape:", df_proc.shape)
print("Final number of columns:", df_proc.shape[1])
print("Min month:", df_proc["month"].min())
print("Max month:", df_proc["month"].max())
print("Unique NUM_POSTE:", df_proc["NUM_POSTE"].nunique())

print("\nFinal missing counts in kept columns (should be all zero):")
display(df_proc.isna().sum().sort_values(ascending=False).head(30))

Duplicate NUM_POSTE-month rows before: 0


/tmp/ipykernel_35427/1331077167.py:127: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: seasonal_impute_for_station_group(g, value_col, q_col))
/tmp/ipykernel_35427/1331077167.py:127: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: seasonal_impute_for_station_group(g, value_col, q_col))
/tmp/ipykernel_35427/1331077167.py:127: FutureWarning: DataFrameGroupBy.apply operated on the grouping


Q columns found: 36
Q/value pairs processed: 36
Q columns skipped because base column missing: 0

Base columns kept: 32
Q columns kept: 4

Imputing retained base numeric columns:
LAT: missing before=0, after=0
LON: missing before=0, after=0
ALTI: missing before=0, after=0
RR: missing before=20923, after=15731
NBRR: missing before=2908, after=312
RRAB: missing before=20895, after=15729
RRABDAT: missing before=23516, after=16412
NBJRR1: missing before=20901, after=15729
NBJRR5: missing before=20901, after=15729
NBJRR10: missing before=20901, after=15729
NBJRR30: missing before=23312, after=17099
NBJRR50: missing before=23312, after=17099
NBJRR100: missing before=23312, after=17099
NBPMERM: missing before=2846, after=284
NBTX: missing before=3167, after=543
NBTN: missing before=3174, after=543
NBTAMPLI: missing before=3184, after=542
NBTM: missing before=3223, after=548
NBTMM: missing before=2854, after=284
NBUN: missing before=3067, after=329
NBUX: missing before=2931, after=328
NBUM: m

NUM_POSTE           0
code_departement    0
month               0
TN                  0
TX                  0
LAT                 0
LON                 0
ALTI                0
RR                  0
NBRR                0
RRAB                0
RRABDAT             0
NBJRR1              0
NBJRR5              0
NBJRR10             0
NBJRR30             0
NBJRR50             0
NBJRR100            0
NBPMERM             0
NBTX                0
NBTN                0
NBTAMPLI            0
NBTM                0
NBTMM               0
NBUN                0
NBUX                0
NBUM                0
NBFXI               0
NBFXY               0
NBFFM               0
dtype: int64

In [8]:
# Cell 3 — Add temporal features for the LSTM

import numpy as np
import pandas as pd

df_proc = df_proc.copy()

# Safety checks
if "month" not in df_proc.columns:
    raise ValueError("month column is missing. Run the cleaning cell first.")
if "NUM_POSTE" not in df_proc.columns:
    raise ValueError("NUM_POSTE column is missing.")

# Ensure sorting
df_proc = df_proc.sort_values(["NUM_POSTE", "month"]).reset_index(drop=True)

# Extract month number from Period[M]
df_proc["mois"] = df_proc["month"].astype(str).str.slice(5, 7).astype(int)

# Cyclical month encoding
df_proc["month_sin"] = np.sin(2 * np.pi * (df_proc["mois"] - 1) / 12)
df_proc["month_cos"] = np.cos(2 * np.pi * (df_proc["mois"] - 1) / 12)

# Long-term trend index from 2001-01 onward
df_proc["time_idx"] = (
    (df_proc["month"].dt.year - 2001) * 12
    + (df_proc["month"].dt.month - 1)
).astype(int)

# Normalize over the full modelling span
max_time_idx = int(df_proc["time_idx"].max())
df_proc["time_idx_norm"] = df_proc["time_idx"] / max_time_idx

print("df_proc shape:", df_proc.shape)
print("Min month:", df_proc["month"].min())
print("Max month:", df_proc["month"].max())
print("Min time_idx:", df_proc["time_idx"].min())
print("Max time_idx:", df_proc["time_idx"].max())
print("Unique months:", df_proc["month"].nunique())
print("Unique NUM_POSTE:", df_proc["NUM_POSTE"].nunique())

df_proc[[
    "NUM_POSTE", "month", "mois",
    "month_sin", "month_cos",
    "time_idx", "time_idx_norm"
]].head()

df_proc shape: (383239, 41)
Min month: 2001-01
Max month: 2025-12
Min time_idx: 0
Max time_idx: 299
Unique months: 300
Unique NUM_POSTE: 2520


,NUM_POSTE,month,mois,month_sin,month_cos,time_idx,time_idx_norm
0,10002001,2001-02,2,0.500000,8.660254e-01,1,0.003344
1,10002001,2001-03,3,0.866025,5.000000e-01,2,0.006689
2,10002001,2002-09,9,-0.866025,-5.000000e-01,20,0.066890
3,10002001,2002-10,10,-1.000000,-1.836970e-16,21,0.070234
4,10002001,2003-03,3,0.866025,5.000000e-01,26,0.086957


In [10]:
df_proc.columns

Index(['NUM_POSTE', 'code_departement', 'month', 'TMM', 'LAT', 'LON', 'ALTI',
       'RR', 'NBRR', 'RRAB', 'RRABDAT', 'NBJRR1', 'NBJRR5', 'NBJRR10',
       'NBJRR30', 'NBJRR50', 'NBJRR100', 'NBPMERM', 'NBTX', 'NBTN', 'NBTAMPLI',
       'NBTM', 'NBTMM', 'NBUN', 'NBUX', 'NBUM', 'NBFXI', 'NBFXY', 'NBFFM',
       'NBINST', 'NBGLOT', 'NBDIFT', 'NBDIRT', 'NBHNEIGEF', 'QRR', 'QRRAB',
       'mois', 'month_sin', 'month_cos', 'time_idx', 'time_idx_norm'],
      dtype='object')

In [8]:
# Cell 4 — Normalize numeric columns on train period, build 36->24 samples, and save tensors

import numpy as np
import pandas as pd
import torch
from pathlib import Path

df_proc = df_proc.copy()

# -----------------------------
# Configuration
# -----------------------------
INPUT_LEN = 36
OUTPUT_LEN = 24
TOTAL_LEN = INPUT_LEN + OUTPUT_LEN

target_cols = ["TN", "TX"]   # monthly minimum and maximum temperature
start_target_year = 2004     # first year we can predict with 36 months of history
# For 24-month forecasts, to avoid leakage across splits:
# - train targets must end by 2021
# - val = 2022-2023
# - test = 2024-2025
# Forecast start years:
#   2022 -> predicts 2022-2023
#   2024 -> predicts 2024-2025
train_last_start_year = 2020

# Safety checks
for tc in target_cols:
    if tc not in df_proc.columns:
        raise ValueError(f"Target column {tc} not found.")
if "month" not in df_proc.columns:
    raise ValueError("month column missing.")
if "NUM_POSTE" not in df_proc.columns:
    raise ValueError("NUM_POSTE column missing.")
if "code_departement" not in df_proc.columns:
    raise ValueError("code_departement column missing.")

# -----------------------------
# 1) Encode categorical IDs
# -----------------------------
df_proc["NUM_POSTE"] = df_proc["NUM_POSTE"].astype("string")
df_proc["code_departement"] = df_proc["code_departement"].astype("string")

station_values = sorted(df_proc["NUM_POSTE"].dropna().unique().tolist())
dept_values = sorted(df_proc["code_departement"].dropna().unique().tolist())

station_to_id = {v: i for i, v in enumerate(station_values)}
dept_to_id = {v: i for i, v in enumerate(dept_values)}

df_proc["station_id"] = df_proc["NUM_POSTE"].map(station_to_id).astype(int)
df_proc["dept_id"] = df_proc["code_departement"].map(dept_to_id).astype(int)

num_stations = len(station_to_id)
num_departments = len(dept_to_id)

print("Stations:", num_stations)
print("Departments:", num_departments)

# -----------------------------
# 2) Define numeric feature columns
#    Keep all numeric columns except helpers / IDs we do not want as features.
#    IMPORTANT: TN/TX are kept as input features, but will NOT be normalized.
# -----------------------------
exclude_numeric = {
    "time_idx",    # raw trend index not used
    "mois",        # month integer not used because we use sin/cos
    "station_id",  # use embedding instead
    "dept_id",     # use embedding instead
}

numeric_feature_cols = [
    c for c in df_proc.columns
    if pd.api.types.is_numeric_dtype(df_proc[c]) and c not in exclude_numeric
]

# Ensure targets are included as input features
for tc in target_cols:
    if tc not in numeric_feature_cols:
        numeric_feature_cols.append(tc)

# Stable order
numeric_feature_cols = [c for c in df_proc.columns if c in numeric_feature_cols]

print("\nNumber of numeric feature columns:", len(numeric_feature_cols))
print("First 25 numeric features:")
print(numeric_feature_cols[:25])

# -----------------------------
# 3) Normalize numeric columns using train period only
#    IMPORTANT: do NOT normalize TN/TX
#    Train period = all rows up to and including 2021
# -----------------------------
train_mask = df_proc["month"].dt.year <= 2021

norm_stats = {}
cols_to_normalize = [c for c in numeric_feature_cols if c not in target_cols]

for col in cols_to_normalize:
    vals = pd.to_numeric(df_proc.loc[train_mask, col], errors="coerce")
    mean = float(vals.mean())
    std = float(vals.std(ddof=0))
    if not np.isfinite(std) or std == 0.0:
        std = 1.0

    df_proc[col] = (pd.to_numeric(df_proc[col], errors="coerce") - mean) / std
    norm_stats[col] = {"mean": mean, "std": std}

print("\nNormalization complete.")
print("Example stats:")
for c in cols_to_normalize[:10]:
    print(c, norm_stats[c])

# -----------------------------
# 4) Build yearly-aligned 36->24 samples
#    Forecast start year = first year of the 24-month target
#    Example:
#      input  = Jan(start_year-3) ... Dec(start_year-1)   [36 months]
#      target = Jan(start_year)   ... Dec(start_year+1)   [24 months]
#
#    Split rules:
#      train -> start_year <= 2020
#      val   -> start_year == 2022   (predicts 2022-2023)
#      test  -> start_year == 2024   (predicts 2024-2025)
#
#    We intentionally skip 2021 and 2023 because they would cross split boundaries.
# -----------------------------
samples = {
    "train": [],
    "val": [],
    "test": [],
}

skipped = {
    "train": 0,
    "val": 0,
    "test": 0,
}

def split_for_start_year(y):
    if y <= train_last_start_year:
        return "train"
    if y == 2022:
        return "val"
    if y == 2024:
        return "test"
    return None  # skip years that would leak across split boundaries

# Work station by station
for num_poste, g in df_proc.groupby("NUM_POSTE", sort=True):
    g = g.sort_values("month").copy()
    g = g.set_index("month")

    dept_id = int(g["dept_id"].iloc[0])
    station_id = int(g["station_id"].iloc[0])

    # latest possible forecast start year is 2024,
    # because we need target through Dec 2025
    for forecast_start_year in range(start_target_year, 2025):
        split = split_for_start_year(forecast_start_year)
        if split is None:
            continue

        input_start_year = forecast_start_year - 3

        input_months = pd.period_range(
            start=pd.Period(f"{input_start_year}-01", freq="M"),
            periods=INPUT_LEN,
            freq="M"
        )
        target_months = pd.period_range(
            start=pd.Period(f"{forecast_start_year}-01", freq="M"),
            periods=OUTPUT_LEN,
            freq="M"
        )
        window_months = input_months.append(target_months)

        # Require the full 60-month span to exist
        window = g.reindex(window_months)

        if window.isna().any().any():
            skipped[split] += 1
            continue

        X_num = window.iloc[:INPUT_LEN][numeric_feature_cols].to_numpy(dtype=np.float32)
        y = window.iloc[INPUT_LEN:][target_cols].to_numpy(dtype=np.float32)  # [24, 2]

        samples[split].append({
            "X_num": X_num,
            "y": y,
            "station_id": station_id,
            "dept_id": dept_id,
            "NUM_POSTE": num_poste,
            "forecast_start_year": forecast_start_year,
            "input_start": str(input_months[0]),
            "target_start": str(target_months[0]),
            "target_end": str(target_months[-1]),
        })

# -----------------------------
# 5) Convert to tensors and save
# -----------------------------
out_dir = Path("wd/src/lstm/trnsors")
out_dir.mkdir(parents=True, exist_ok=True)

def pack_and_save(split_name, records):
    if len(records) == 0:
        raise ValueError(f"No samples found for split: {split_name}")

    X_num = torch.tensor(np.stack([r["X_num"] for r in records]), dtype=torch.float32)
    y = torch.tensor(np.stack([r["y"] for r in records]), dtype=torch.float32)  # [N, 24, 2]
    station_id = torch.tensor([r["station_id"] for r in records], dtype=torch.long)
    dept_id = torch.tensor([r["dept_id"] for r in records], dtype=torch.long)

    meta = [
        {
            "NUM_POSTE": r["NUM_POSTE"],
            "forecast_start_year": r["forecast_start_year"],
            "input_start": r["input_start"],
            "target_start": r["target_start"],
            "target_end": r["target_end"],
        }
        for r in records
    ]

    payload = {
        "X_num": X_num,
        "station_id": station_id,
        "dept_id": dept_id,
        "y": y,
        "meta": meta,
        "numeric_feature_cols": numeric_feature_cols,
        "norm_stats": norm_stats,
        "station_to_id": station_to_id,
        "dept_to_id": dept_to_id,
        "input_len": INPUT_LEN,
        "output_len": OUTPUT_LEN,
        "target_cols": target_cols,
    }

    file_path = out_dir / f"{split_name}_tn_tx_24_mo.pt"
    torch.save(payload, file_path)

    print(f"{split_name}:")
    print(f"  X_num shape: {tuple(X_num.shape)}")
    print(f"  y shape:     {tuple(y.shape)}")
    print(f"  station_id:  {tuple(station_id.shape)}")
    print(f"  dept_id:     {tuple(dept_id.shape)}")
    print(f"  saved to:    {file_path}")

    return payload

train_pack = pack_and_save("train", samples["train"])
val_pack = pack_and_save("val", samples["val"])
test_pack = pack_and_save("test", samples["test"])

print("\nSkipped windows:")
print(skipped)

print("\nDone.")
print("Train samples:", len(samples["train"]))
print("Val samples:", len(samples["val"]))
print("Test samples:", len(samples["test"]))
print("Total samples:", len(samples["train"]) + len(samples["val"]) + len(samples["test"]))

Stations: 3530
Departments: 95

Number of numeric feature columns: 36
First 25 numeric features:
['TN', 'TX', 'LAT', 'LON', 'ALTI', 'RR', 'NBRR', 'RRAB', 'RRABDAT', 'NBJRR1', 'NBJRR5', 'NBJRR10', 'NBJRR30', 'NBJRR50', 'NBJRR100', 'NBPMERM', 'NBTX', 'NBTN', 'NBTAMPLI', 'NBTM', 'NBTMM', 'NBUN', 'NBUX', 'NBUM', 'NBFXI']

Normalization complete.
Example stats:
LAT {'mean': 46.075231778856484, 'std': 2.0730986664775832}
LON {'mean': 3.0112507719379265, 'std': 2.8436743914061307}
ALTI {'mean': 389.11123221873265, 'std': 421.2386167421264}
RR {'mean': 75.82688342878258, 'std': 55.79659231735702}
NBRR {'mean': 30.185457232588213, 'std': 2.3767648821215497}
RRAB {'mean': 22.346701274709027, 'std': 17.847737953709498}
RRABDAT {'mean': 15.300032329576945, 'std': 9.155143979062188}
NBJRR1 {'mean': 9.494297062627009, 'std': 4.659151864105679}
NBJRR5 {'mean': 4.577326805837798, 'std': 3.081969738470699}
NBJRR10 {'mean': 2.3425512654720118, 'std': 2.1108884436698765}
train:
  X_num shape: (29908, 36,

In [9]:
# Cell 5 — Verify saved tensors against source dataframe

import numpy as np
import pandas as pd
import torch
from pathlib import Path
import random

# -----------------------------
# Load saved payloads
# -----------------------------
out_dir = Path("wd/src/lstm/trnsors")

train_pack = torch.load(out_dir / "train_tn_tx_24_mo.pt", map_location="cpu")
val_pack   = torch.load(out_dir / "val_tn_tx_24_mo.pt", map_location="cpu")
test_pack  = torch.load(out_dir / "test_tn_tx_24_mo.pt", map_location="cpu")

print("Loaded:")
print("  train:", tuple(train_pack["X_num"].shape), tuple(train_pack["y"].shape))
print("  val:  ", tuple(val_pack["X_num"].shape), tuple(val_pack["y"].shape))
print("  test: ", tuple(test_pack["X_num"].shape), tuple(test_pack["y"].shape))

# -----------------------------
# Basic size checks
# -----------------------------
n_num_features = train_pack["X_num"].shape[-1]
input_len = train_pack["input_len"]
output_len = train_pack["output_len"]

station_emb_dim = 64
dept_emb_dim = 16
full_input_dim = n_num_features + station_emb_dim + dept_emb_dim

print("\nFeature dimensions:")
print("  numeric features only:", n_num_features)
print("  station embedding dim:", station_emb_dim)
print("  dept embedding dim:   ", dept_emb_dim)
print("  expected model input dim after concat:", full_input_dim)

assert input_len == 36, f"Expected input_len=36, got {input_len}"
assert output_len == 24, f"Expected output_len=24, got {output_len}"

# -----------------------------
# Helper for comparing one random sample against df_proc
# -----------------------------
target_cols = train_pack["target_cols"]

def compare_sample(pack, split_name, n_checks=3):
    X_num = pack["X_num"]
    y = pack["y"]
    meta = pack["meta"]
    numeric_feature_cols = pack["numeric_feature_cols"]

    print(f"\n=== {split_name.upper()} SPLIT CHECK ===")
    print("Samples:", X_num.shape[0])

    rng = random.Random(42)
    sample_indices = rng.sample(range(len(meta)), k=min(n_checks, len(meta)))

    for idx in sample_indices:
        m = meta[idx]
        station = m["NUM_POSTE"]
        input_start = pd.Period(m["input_start"], freq="M")
        target_start = pd.Period(m["target_start"], freq="M")
        target_end = pd.Period(m["target_end"], freq="M")

        input_months = pd.period_range(start=input_start, periods=input_len, freq="M")
        target_months = pd.period_range(start=target_start, periods=output_len, freq="M")

        g = df_proc[df_proc["NUM_POSTE"].astype(str) == str(station)].sort_values("month").copy()
        g = g.set_index("month")

        # Rebuild source window
        src_input = g.reindex(input_months)[numeric_feature_cols].to_numpy(dtype=np.float32)
        src_target = g.reindex(target_months)[target_cols].to_numpy(dtype=np.float32)

        tensor_input = X_num[idx].numpy()
        tensor_target = y[idx].numpy()

        input_ok = np.allclose(tensor_input, src_input, equal_nan=True)
        target_ok = np.allclose(tensor_target, src_target, equal_nan=True)

        print(f"\nSample idx: {idx}")
        print("  NUM_POSTE:", m["NUM_POSTE"])
        print("  forecast_start_year:", m["forecast_start_year"])
        print("  input_start:", m["input_start"], "target:", m["target_start"], "->", m["target_end"])
        print("  station_id:", int(pack["station_id"][idx]), "dept_id:", int(pack["dept_id"][idx]))
        print("  input match:", input_ok)
        print("  target match:", target_ok)

        # Show a few direct value comparisons
        check_cols = random.sample(numeric_feature_cols, k=min(5, len(numeric_feature_cols)))
        print("  Direct value checks:")
        for col in check_cols:
            col_idx = numeric_feature_cols.index(col)
            month0 = input_months[0]
            t_val = tensor_input[0, col_idx]
            s_val = src_input[0, col_idx]
            print(f"    {month0} | {col}: tensor={t_val:.6f} | source={s_val:.6f}")

        # Also show first target month for sanity
        print(f"  First target month {target_months[0]}:")
        for j, tc in enumerate(target_cols):
            print(f"    {tc} tensor={tensor_target[0, j]:.6f} | source={src_target[0, j]:.6f}")

        assert input_ok, f"Input mismatch for sample {idx}"
        assert target_ok, f"Target mismatch for sample {idx}"

# -----------------------------
# Run checks for all splits
# -----------------------------
compare_sample(train_pack, "train", n_checks=3)
compare_sample(val_pack, "val", n_checks=2)
compare_sample(test_pack, "test", n_checks=2)

print("\nAll checks passed.")

Loaded:
  train: (29908, 36, 36) (29908, 24, 2)
  val:   (1379, 36, 36) (1379, 24, 2)
  test:  (1552, 36, 36) (1552, 24, 2)

Feature dimensions:
  numeric features only: 36
  station embedding dim: 64
  dept embedding dim:    16
  expected model input dim after concat: 116

=== TRAIN SPLIT CHECK ===
Samples: 29908

Sample idx: 20952
  NUM_POSTE: 66206004
  forecast_start_year: 2012
  input_start: 2009-01 target: 2012-01 -> 2013-12
  station_id: 2457 dept_id: 65
  input match: True
  target match: True
  Direct value checks:
    2009-01 | NBUX: tensor=-0.830431 | source=-0.830431
    2009-01 | QRRAB: tensor=-0.004901 | source=-0.004901
    2009-01 | RRABDAT: tensor=1.714879 | source=1.714879
    2009-01 | NBJRR100: tensor=-0.080878 | source=-0.080878
    2009-01 | NBTX: tensor=0.360980 | source=0.360980
  First target month 2012-01:
    TN tensor=2.100000 | source=2.100000
    TX tensor=13.100000 | source=13.100000

Sample idx: 3648
  NUM_POSTE: 19146001
  forecast_start_year: 2006
  in

In [10]:
# Cell 6 — Build app_24.pt for 2026-2027 inference inputs only
# Input window: 2023-01 to 2025-12
# Output months (predicted): 2026-01 to 2027-12

import numpy as np
import pandas as pd
import torch
from pathlib import Path

df_proc = df_proc.copy()

INPUT_LEN = 36
OUTPUT_LEN = 24
forecast_start = pd.Period("2026-01", freq="M")
forecast_end = pd.Period("2027-12", freq="M")
forecast_year = 2026

# -----------------------------
# Safety checks
# -----------------------------
for c in ["month", "NUM_POSTE", "code_departement"]:
    if c not in df_proc.columns:
        raise ValueError(f"Missing column: {c}")

# Use the same feature/mapping objects created in the training cell
if "numeric_feature_cols" not in globals():
    raise ValueError("numeric_feature_cols not found. Run the training sample-building cell first.")
if "station_to_id" not in globals():
    raise ValueError("station_to_id not found. Run the training sample-building cell first.")
if "dept_to_id" not in globals():
    raise ValueError("dept_to_id not found. Run the training sample-building cell first.")

# -----------------------------
# Build the 36-month input window for 2026-2027 prediction
# -----------------------------
input_start = pd.Period("2023-01", freq="M")
input_months = pd.period_range(start=input_start, periods=INPUT_LEN, freq="M")
forecast_months = pd.period_range(start=forecast_start, periods=OUTPUT_LEN, freq="M")

app_records = []
skipped = 0

df_proc = df_proc.sort_values(["NUM_POSTE", "month"]).reset_index(drop=True)

for num_poste, g in df_proc.groupby("NUM_POSTE", sort=True):
    g = g.sort_values("month").copy()
    g = g.set_index("month")

    num_poste_str = str(num_poste)
    if num_poste_str not in station_to_id:
        skipped += 1
        continue

    dept_value = str(g["code_departement"].iloc[0])
    if dept_value not in dept_to_id:
        skipped += 1
        continue

    window = g.reindex(input_months)

    # Need the full history window to predict 2026-2027
    if window.isna().any().any():
        skipped += 1
        continue

    X_num = window[numeric_feature_cols].to_numpy(dtype=np.float32)

    app_records.append({
        "X_num": X_num,
        "station_id": int(station_to_id[num_poste_str]),
        "dept_id": int(dept_to_id[dept_value]),
        "NUM_POSTE": num_poste_str,
        "DEP": dept_value,
        "input_start": str(input_months[0]),
        "input_end": str(input_months[-1]),
        "forecast_start": str(forecast_months[0]),
        "forecast_end": str(forecast_months[-1]),
    })

print("App records:", len(app_records))
print("Skipped stations:", skipped)

if len(app_records) == 0:
    raise ValueError("No app records were built. Check that 2023-01 to 2025-12 exists for each station.")

# -----------------------------
# Pack tensors and save
# -----------------------------
X_num = torch.tensor(np.stack([r["X_num"] for r in app_records]), dtype=torch.float32)
station_id = torch.tensor([r["station_id"] for r in app_records], dtype=torch.long)
dept_id = torch.tensor([r["dept_id"] for r in app_records], dtype=torch.long)

meta = [
    {
        "NUM_POSTE": r["NUM_POSTE"],
        "DEP": r["DEP"],
        "input_start": r["input_start"],
        "input_end": r["input_end"],
        "forecast_start": r["forecast_start"],
        "forecast_end": r["forecast_end"],
    }
    for r in app_records
]

app_pack = {
    "X_num": X_num,
    "station_id": station_id,
    "dept_id": dept_id,
    "meta": meta,
    "numeric_feature_cols": numeric_feature_cols,
    "station_to_id": station_to_id,
    "dept_to_id": dept_to_id,
    "input_len": INPUT_LEN,
    "output_len": OUTPUT_LEN,
    "forecast_year": forecast_year,
    "forecast_months": [str(p) for p in forecast_months],
}

out_dir = Path("wd/src/lstm/trnsors")
out_dir.mkdir(parents=True, exist_ok=True)

torch.save(app_pack, out_dir / "app_24.pt")

print("Saved:", out_dir / "app_24.pt")
print("X_num shape:", tuple(X_num.shape))
print("station_id shape:", tuple(station_id.shape))
print("dept_id shape:", tuple(dept_id.shape))

App records: 1746
Skipped stations: 1784
Saved: wd/src/lstm/trnsors/app_24.pt
X_num shape: (1746, 36, 36)
station_id shape: (1746,)
dept_id shape: (1746,)
